<a href="https://colab.research.google.com/github/joyhwp/personal_projects/blob/main/07_LLM_Engineering_Sessions/%ED%8C%8C%EC%9D%B8%ED%8A%9C%EB%8B%9D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

사용 패키지 버전

* datasets 4.0.0
* transformers 4.57.3
* bitsandbytes 0.49.0
* trl 0.16.1

*trl 버전 0.17에서 SFTTrainer를 실행할 때 KeyError: 'completion'가 발생하므로 trl 버전을 0.16.1로 고정한다.*

In [ ]:
%%capture
!pip install bitsandbytes trl==0.16.1

## 지도 학습 미세 튜닝

### 데이터 전처리

In [ ]:
from transformers import AutoTokenizer
from datasets import load_dataset


# 채팅 템플릿을 사용하기 위해 토크나이저를 로드합니다.
template_tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")

def format_prompt(example):
    """TinyLlama의 <|user|> 템플릿으로 프롬프트를 포맷팅합니다"""

    # 채팅 템플릿 구성
    chat = example["messages"]
    prompt = template_tokenizer.apply_chat_template(chat, tokenize=False)

    return {"text": prompt}

# 데이터를 로드하고 TinyLlama 템플릿을 적용합니다.
dataset = (
    load_dataset("HuggingFaceH4/ultrachat_200k",  split="test_sft")
      .shuffle(seed=42)
      .select(range(500))
)
dataset = dataset.map(format_prompt).remove_columns(['messages'])

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

...

Generating train_sft split:   0%|          | 0/207865 [00:00<?, ? examples/s]

Generating test_sft split:   0%|          | 0/23110 [00:00<?, ? examples/s]

Generating train_gen split:   0%|          | 0/256032 [00:00<?, ? examples/s]

Generating test_gen split:   0%|          | 0/28304 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
# 프롬프트 예시
print(dataset["text"][123])

<|user|>
Here is a piece of text: COWPENS, SC-- George Thomas Spencer, Jr., 87, of Cowpens, died Sunday, February 27, 2011, at White Oak Estates. Born May 31, 1923, in Cowpens, he was the son of the late George Thomas, Sr. And Lorene Smith Spencer.
Mr. Spencer was a U.S. Marines veteran of World War II and was awarded the Purple Heart. He was a member of Mountain View Baptist Church and a truck driver for General Motors Cadillac Division.
Survivors include his wife, Peggy Coleman Spencer; three sons, Wayne (Mary Sue) Spencer of Chesnee, Randy (Pam) Spencer of Gaffney, and Timothy Spencer of Cowpens; a daughter, Kathy (Johnny) Mullinax of Boiling Springs; 14 grandchildren; 19 great-grandchildren; and a sister, Bonnie Carter of Spartanburg. He was predeceased by a son, Michael Spencer, and a daughter-in-law, Renee Spencer.
Visitation will be 2:30 PM until 3:45 PM Wednesday, March 2, 2011, at Mountain View Baptist Church. Funeral services will follow at 4:00 PM at the church, conducted by

### 모델 양자화

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

# 4-비트 양자화 설정 - QLoRA의 Q 단계
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,  # 4-비트 정밀도 모델 로드
    bnb_4bit_quant_type="nf4",  # 양자화 종류
    bnb_4bit_compute_dtype="float16",  # 계산 dtype
    bnb_4bit_use_double_quant=True,  # 이중 양자화 적용
)

# 모델을 로드하고 GPU에서 훈련합니다.
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",

    # 일반적인 SFT에서는 다음을 삭제하세요.
    quantization_config=bnb_config,
)
model.config.use_cache = False

# LLaMA 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = "<PAD>"
tokenizer.padding_side = "left"

config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/4.40G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

### 설정

#### LoRA 설정(하이퍼파라미터 설정)

In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

# LoRA 설정
peft_config = LoraConfig(
    lora_alpha=32,  # LoRA 스케일링
    lora_dropout=0.1,  # LoRA 층의 드롭아웃
    r=16,  # 압축된 행렬의 랭크. 이 값을 증가시키면 덜 압축되고 표현성능도 향상됩니다. 일반적으로 4~64
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=  # 대상 층
     ['q_proj', 'k_proj', 'v_proj', 'o_proj']
)

# 훈련을 위한 모델 준비
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)

#### 훈련 설정

In [ ]:
from trl import SFTConfig

output_dir = "./results"

# 훈련 매개변수
training_arguments = SFTConfig(
    output_dir=output_dir,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit", # 옵티마이저
    learning_rate=2e-4, # 학습률
    lr_scheduler_type="cosine", # 스케줄러 타입
    num_train_epochs=1, # 훈련 반복 횟수
    logging_steps=10,
    fp16=True,
    gradient_checkpointing=True,
    dataset_text_field="text",
    max_length=256
)

### 훈련

In [ ]:
from trl import SFTTrainer

# 지도 미세 튜닝 매개변수 지정
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    processing_class=tokenizer,
    args=training_arguments,

    # 일반적인 SFT에서는 다음을 삭제하세요.
    peft_config=peft_config,
)

# 모델 훈련
trainer.train()

# QLoRA 가중치 저장
trainer.model.save_pretrained("TinyLlama-1.1B-qlora")

Converting train dataset to ChatML:   0%|          | 0/500 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 0}.


Step,Training Loss
10,2.007414
20,1.643856
30,1.656591
40,1.651965
50,1.588676
60,1.637905


### 어댑터 병합

QLoRA 가중치를 훈련한 후 이를 사용하려면 원본 가중치와 병합해야 합니다

In [ ]:
from peft import AutoPeftModelForCausalLM

model = AutoPeftModelForCausalLM.from_pretrained(
    "TinyLlama-1.1B-qlora",
    low_cpu_mem_usage=True,
    device_map="auto",
)

# LoRA와 베이스 모델을 병합합니다.
merged_model = model.merge_and_unload()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

### 추론

In [ ]:
from transformers import pipeline

# 사전에 정의된 프롬프트 템플릿을 사용합니다.
prompt = """<|user|>
Tell me something about Large Language Models.</s>
<|assistant|>
"""

# 인스트럭션 튜닝된 모델을 실행합니다.
pipe = pipeline(task="text-generation", model=merged_model, tokenizer=tokenizer)
print(pipe(prompt)[0]["generated_text"])

<|user|>
Tell me something about Large Language Models.</s>
<|assistant|>
Large Language Models are neural networks that learn to model words and phrases in a natural language. These models can be used to generate natural language text, such as summaries or articles, or to predict the next word in a sequence. Large Language Models have been increasingly used in Natural Language Processing applications, such as in chatbots and machine translation.

Large Language Models are implemented in a variety of methods, including recurrent neural networks, transformer models, and convolutional neural networks. Recurrent neural networks are commonly used for language modeling applications, because they can capture long-term dependencies in text. Transformer models have been shown to be particularly effective at language modeling, due to their ability to capture contextual information. Transformer models are often used for Natural Language Generation applications, such as summarization or article g